# MLC LLM + Cloudflare Tunnel on Google Colab 🚀

Bu notebook, **MLC LLM**'i Google Colab'ın ücretsiz **T4 GPU**'sunda çalıştırır
ve **Cloudflare Tunnel** ile dışarıya açar. Render'daki backend bu tünel üzerinden
MLC LLM'e bağlanır.

---

## Mimari
```
Render (backend)
    ↓  (HTTP)
Cloudflare Tunnel (public URL)
    ↓  (HTTP)
Google Colab (T4 GPU)
    ├── MLC LLM (CUDA)
    └── Gemma-2B (q4f16)
```

## Kullanım
1. **Runtime → Change runtime type** → **T4 GPU** seçin
2. Tüm hücreleri sırayla çalıştırın (Ctrl+F9)
3. Son hücredeki **Tunnel URL**'i kopyalayın
4. Render'da `MLC_LLM_BASE_URL` ve `MLC_LLM_ENABLED=true` ayarlayın

## 1. GPU Kontrolü
T4 GPU'nun erişilebilir olduğunu doğrulayalım.

In [ ]:
!nvidia-smi

## 2. MLC LLM Kurulumu
CUDA 12.1 için nightly wheels kuruluyor.

In [ ]:
import sys, os, time, re, threading, subprocess, urllib.request, json, signal, atexit, textwrap

# MLC LLM CUDA 12.1 wheels (T4 GPU için)
!pip install --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cu121 mlc-ai-nightly-cu121 2>&1 | tail -5

print("✅ MLC LLM kurulumu tamamlandı")

## 3. Cloudflare Tunnel Kurulumu
cloudflared binary'si indiriliyor.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

## 4. MLC LLM Sunucusunu Başlat
Gemma-2B modeli indirilip MLC LLM REST API sunucusu başlatılıyor.

⏱ **Model indirme + derleme ~3-5 dakika sürebilir.**

In [ ]:
MODEL = "HF://mlc-ai/gemma-2b-it-q4f16_1-MLC"
HOST = "0.0.0.0"
PORT = 8000

mlc_process = None
server_ready = threading.Event()

def start_mlc_server():
    global mlc_process
    cmd = [sys.executable, "-m", "mlc_llm", "serve", MODEL, "--host", HOST, "--port", str(PORT)]
    print(f"🚀 Starting MLC LLM server...")
    print(f"   {' '.join(cmd)}")
    mlc_process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in mlc_process.stdout:
        print(line, end='')
        if "Uvicorn running on" in line or "Application startup complete" in line or "listening" in line.lower():
            server_ready.set()
            print("\n✅ MLC LLM server is ready!\n")

threading.Thread(target=start_mlc_server, daemon=True).start()

# Sunucunun hazır olmasını bekle (timeout: 600 sn = 10 dk)
print("⏳ Waiting for MLC LLM server to start (model download + compile)...")
for i in range(300):
    if server_ready.is_set():
        break
    try:
        req = urllib.request.Request(f"http://{HOST}:{PORT}/v1/models")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                print("✅ MLC LLM server is ready!")
                server_ready.set()
                break
    except Exception:
        pass
    if i % 10 == 0:
        print(f"  Waiting... ({i*2}s)")
    time.sleep(2)

if not server_ready.is_set():
    print("⚠️ Server did not signal ready. It may still be starting...")
else:
    print(f"✅ MLC LLM ready at http://{HOST}:{PORT}")

## 5. Cloudflare Tunnel Başlat
MLC LLM sunucusunu internet üzerinden erişilebilir yapıyoruz.

In [ ]:
tunnel_process = None
tunnel_url = None
url_lock = threading.Lock()

def start_tunnel():
    global tunnel_url, tunnel_process
    tunnel_process = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )

    url_pat = re.compile(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com')

    def read_output(stream, label):
        global tunnel_url
        for line in stream:
            print(f"[{label}] {line}", end='')
            m = url_pat.search(line)
            if m:
                with url_lock:
                    if not tunnel_url:
                        tunnel_url = m.group(0)
                        print(f"\n{'='*60}")
                        print(f"✅ TUNNEL URL: {tunnel_url}")
                        print(f"{'='*60}\n")

    threading.Thread(target=read_output, args=(tunnel_process.stdout, 'cloudflared'), daemon=True).start()
    threading.Thread(target=read_output, args=(tunnel_process.stderr, 'cloudflared'), daemon=True).start()

print("🌐 Starting Cloudflare Tunnel...")
start_tunnel()

# URL'nin gelmesini bekle (max 60 sn)
for i in range(60):
    with url_lock:
        if tunnel_url:
            break
    time.sleep(1)

if tunnel_url:
    print(f"\n✅ Tunnel is active: {tunnel_url}")
else:
    print("⚠️ Tunnel URL not detected. Check output above.")

## 6. API Testi
Tünel üzerinden MLC LLM'e istek gönderip çalıştığını doğrulayalım.

In [ ]:
if tunnel_url:
    test_payload = {
        "model": "default",
        "messages": [
            {"role": "user", "content": "Merhaba, 1+1 kaç eder? Sadece rakamla cevap ver."}
        ],
        "max_tokens": 10
    }

    req = urllib.request.Request(
        f"{tunnel_url}/v1/chat/completions",
        data=json.dumps(test_payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            result = json.loads(resp.read())
            content = result['choices'][0]['message']['content']
            print(f"✅ MLC LLM response: {content}")
    except Exception as e:
        print(f"❌ Test failed: {e}")
else:
    print("⚠️ Tunnel URL not available. Run the previous cell first.")

## 7. Render Ayarları

**Render Dashboard'da aşağıdaki env değişkenlerini ayarlayın:**

| Variable | Value |
|---|---|
| `MLC_LLM_ENABLED` | `true` |
| `MLC_LLM_BASE_URL` | `<yukarıdaki Tunnel URL>` |

**Render'da yeniden deploy etmeyi unutmayın!**


## 8. Canlı Durum
Sunucu çalışırken bu hücre açık kalır. Colab oturumu kapanana kadar (~12 saat) servis ayaktadır.
Kapanınca tüm hücreleri tekrar çalıştırmanız gerekir.

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════╗
║              MLC LLM + Cloudflare Tunnel                    ║
╠══════════════════════════════════════════════════════════════╣
║  Model:      {MODEL:<51}║
║  GPU:        T4 (NVIDIA CUDA 12.1)                          ║
║  Local URL:  http://{HOST}:{PORT:<41}║
║  Tunnel URL: {tunnel_url or 'WAITING...':<51}║
╠══════════════════════════════════════════════════════════════╣
║  Render'da ayarlanacak env değişkenleri:                    ║
║                                                              ║
║  MLC_LLM_ENABLED=true                                        ║
║  MLC_LLM_BASE_URL={tunnel_url or '<TUNNEL_URL>':<44}║
╠══════════════════════════════════════════════════════════════╣
║  ⏳ Oturum açık kalana kadar çalışır.                       ║
║  Oturum kapanınca tüm hücreleri yeniden çalıştırın.         ║
╚══════════════════════════════════════════════════════════════╝
""")

# Sunucu çalışırken bekle (kesintisiz)
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")